In [1]:
import pandas as pd

# CSV-Datei einlesen
# Quelle: https://www.landesdatenbank.nrw.de/
df = pd.read_csv("../data/raw/2024/13111-54i.csv", sep=";", encoding="latin1", skiprows=6)

# Überblick:
print(df.head())
print(df.columns)
df.info()


   Unnamed: 0           Unnamed: 1 Unnamed: 2  \
0  31.12.2024  Nordrhein-Westfalen          A   
1  31.12.2024  Nordrhein-Westfalen        B-F   
2  31.12.2024  Nordrhein-Westfalen        B-E   
3  31.12.2024  Nordrhein-Westfalen          C   
4  31.12.2024  Nordrhein-Westfalen          F   

                               Unnamed: 3 Insgesamt ohne Schulabschluss  \
0    Land- und Forstwirtschaft, Fischerei     30831                1307   
1                  Produzierendes Gewerbe   1822337               44219   
2  Produzierendes Gewerbe ohne Baugewerbe   1440080               31212   
3                  Verarbeitendes Gewerbe   1298577               29181   
4                              Baugewerbe    382257               13007   

  mit Volks-/Haupschulabschluss mit mittlerer Reife/gleichwertigem Abschluss  \
0                          5791                                         8057   
1                        394860                                       539457   
2             

In [2]:
#relevante Spalten und Zeilen filtern
df = df[df["Unnamed: 3"] == "Insgesamt"]
df = df.drop(df.columns[[0, 2, 3, -5, -4, -3, -1]], axis=1)

df = df.dropna(axis=1, how="all")
df = df.dropna(axis=0, how="all")
print(df)

                         Unnamed: 1 Insgesamt ohne Schulabschluss  \
13              Nordrhein-Westfalen   7390826              193750   
27     Düsseldorf, Regierungsbezirk   2202648               56807   
41          Düsseldorf, krfr. Stadt    458999                7641   
55            Duisburg, krfr. Stadt    182109                5475   
69               Essen, krfr. Stadt    271129                7753   
..                              ...       ...                 ...   
797                Märkischer Kreis    159599                4737   
811                     Olpe, Kreis     61557                1848   
825      Siegen-Wittgenstein, Kreis    120724                1833   
839                    Soest, Kreis    115755                2365   
853                     Unna, Kreis    137858                4354   

    mit Volks-/Haupschulabschluss  \
13                        1086124   
27                         303824   
41                          34742   
55                     

In [3]:
# Spalten umbenennen
df = df.rename(columns={"Unnamed: 1": "Name"})

# Ints und Strings anpassen
df["Name"] = df["Name"].str.strip()
cols = df.columns.difference(["Name"])
df[cols] = (
     df[cols]
    .apply(pd.to_numeric, errors="coerce")
    .astype("Int64")  
)


# nach Kreisen und kreisfreien Städten filtern 
df = df[df["Name"].str.contains("Kreis|krfr. Stadt|Städteregion", case=False,na=False)]

In [4]:
print(df.head())
print(df.loc[df["ohne Schulabschluss"].isna(), "Name"])


                            Name  Insgesamt  ohne Schulabschluss  \
41       Düsseldorf, krfr. Stadt     458999                 7641   
55         Duisburg, krfr. Stadt     182109                 5475   
69            Essen, krfr. Stadt     271129                 7753   
83          Krefeld, krfr. Stadt      96597                 2817   
97  Mönchengladbach, krfr. Stadt     107113                 3582   

    mit Volks-/Haupschulabschluss  \
41                          34742   
55                          28083   
69                          31409   
83                          14857   
97                          16233   

    mit mittlerer Reife/gleichwertigem Abschluss  mit Abitur/Fachabitur  \
41                                         86146                 268486   
55                                         46895                  71069   
69                                         59551                 134759   
83                                         25037                  40

In [5]:
# Zeile löschen und Name anpassen
#df = df[df["Name"] != "Essen, krfr. Stadt"]
df = df[df["Name"] != "Aachen, Kreis"]
df = df[df["Name"] != "Aachen, krfr. Stadt"]

print(df.columns)

Index(['Name', 'Insgesamt', 'ohne Schulabschluss',
       'mit Volks-/Haupschulabschluss',
       'mit mittlerer Reife/gleichwertigem Abschluss', 'mit Abitur/Fachabitur',
       'mit akademischem Abschluss'],
      dtype='object')


In [6]:
from ipynb.fs.defs.functions import clean_and_sort, validate_df

# Namen bereinigen
df = clean_and_sort(df, 'Insgesamt', 'ohne Schulabschluss',
       'mit Volks-/Haupschulabschluss',
       'mit mittlerer Reife/gleichwertigem Abschluss', 'mit Abitur/Fachabitur',
       'mit akademischem Abschluss')


In [7]:
#Anteil des jew. Abschlusses an Gesamtbevölkerung des Kreises berechnen
#df_bevoelkerung = pd.read_csv("../data/processed/bevoelkerung_2024.csv", sep=",")

#df = df.merge(df_bevoelkerung[["Name", "Gesamtbevölkerung"]], on="Name", how="left", validate="one_to_one")

#print(df.columns)





In [8]:

abschluss_cols = df.columns.drop(["Name", "Insgesamt"])

df[abschluss_cols] = df[abschluss_cols].div(df["Insgesamt"], axis=0)
#df.drop(["Gesamtbevölkerung"])
print(df.iloc[[2]])

    Name  Insgesamt  ohne Schulabschluss  mit Volks-/Haupschulabschluss  \
2  Essen     271129             0.028595                       0.115845   

   mit mittlerer Reife/gleichwertigem Abschluss  mit Abitur/Fachabitur  \
2                                      0.219641               0.497029   

   mit akademischem Abschluss  
2                      0.2311  


In [9]:

weights = {
    'ohne Schulabschluss':0,
    'mit Volks-/Haupschulabschluss':1,
    'mit mittlerer Reife/gleichwertigem Abschluss':2,
    'mit Abitur/Fachabitur':3,
    'mit akademischem Abschluss':4
}

df["Bildungsindex"] = sum(
    df[col] * weight
    for col, weight in weights.items()
)

#df = df["Name", "Bildungsindex"]

In [10]:
print(df)

                          Name  Insgesamt  ohne Schulabschluss  \
0                   Düsseldorf     458999             0.016647   
1                     Duisburg     182109             0.030064   
2                        Essen     271129             0.028595   
3                      Krefeld      96597             0.029162   
4              Mönchengladbach     107113             0.033441   
5          Mülheim an der Ruhr      64950             0.018938   
6                   Oberhausen      70812             0.033412   
7                    Remscheid      46875             0.031915   
8                     Solingen      52376             0.038758   
9                    Wuppertal     132936              0.02778   
10                       Kleve     107751             0.023935   
11                    Mettmann     204155             0.023551   
12           Rhein-Kreis Neuss     160805             0.024508   
13                     Viersen      97971             0.036286   
14        

In [11]:
df = df[["Name", "Bildungsindex"]]

In [12]:
print(df)

                          Name  Bildungsindex
0                   Düsseldorf       3.489899
1                     Duisburg       2.451916
2                        Essen       2.970615
3                      Krefeld       2.624823
4              Mönchengladbach       2.530925
5          Mülheim an der Ruhr       2.961709
6                   Oberhausen        2.28851
7                    Remscheid       2.494955
8                     Solingen       2.438273
9                    Wuppertal       2.670879
10                       Kleve       2.190922
11                    Mettmann       2.712772
12           Rhein-Kreis Neuss       2.543826
13                     Viersen       2.282349
14                       Wesel       2.325042
15                        Bonn       3.688975
16                        Köln       3.358648
17                  Leverkusen       2.918638
18                      Aachen       3.103331
19                       Düren       2.514448
20            Rhein-Erft-Kreis    

In [13]:
validate_df(df,
    not_null= ["Bildungsindex"],
    positive= ["Bildungsindex"],
    numeric= ["Bildungsindex"],
    key_cols=["Name"])

DataFrame: alle Checks bestanden


[]

In [14]:
# speichern
df.to_csv("../data/processed/bildungsabschluss_2024.csv", index=False)
